# B0005 — improvement round: fixed optimization (A) + closing the information gap (B)

Previous round: best architecture 0.01053, Bayesian optimization 0.01063 — no gain from 60 trials
over 46 hyperparameters. Published range 0.00481-0.02302.

### A — why the optimization failed, and what changes

The search improved validation by 42% while test got slightly worse. Two causes, both fixed here:

| Cause | Fix |
|---|---|
| validation forecast 25 cycles from one origin, test forecast 35 from another — not comparable | **three validation origins, each with the same 35-cycle horizon as test**: 89->123, 94->128, 99->133. Objective is the mean across them. |
| 46 hyperparameters against ~25 validation points | search space cut to **12 hyperparameters** |
| search picked a 1.06M-parameter model for 92 windows | **parameter penalty** added to the objective |

### B — closing the information gap

Tian's models use health indicators engineered from the voltage and temperature curves; ours used
capacity alone. Those indicators cannot simply be fed forward in a recursive forecast, so the model
becomes **multivariate autoregressive**: it predicts every channel one step ahead and feeds the whole
predicted vector back. Cycle index is treated separately — it is genuinely known for future cycles,
so it is supplied as a true value at each step rather than predicted.

| Mode | Inputs | Fed back during forecasting |
|---|---|---|
| F0 | capacity only | capacity (this is the previous round's setup) |
| F1 | capacity + cycle index | capacity predicted; cycle index known |
| F2 | capacity + 8 health indicators | all 9 channels predicted |
| F3 | capacity + 8 indicators + cycle index | 9 predicted, cycle index known |

Architecture is fixed to the previous round's winner — multi-scale CNN + bidirectional LSTM, no
attention, delta target — so any change is attributable to features and tuning, not architecture.

### C — variance check

Every stage reports mean +/- std over seeds and the best seed, so we can see at the end whether the
seed spread still exceeds the published range.

Run from the folder containing `B0005.mat`. Roughly 1-1.5 hours.

In [1]:
import os, json, gc, time
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
import numpy as np, pandas as pd
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras import regularizers
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Dropout, Dense, LSTM,
                                     Bidirectional, Concatenate, BatchNormalization,
                                     GlobalAveragePooling1D)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import keras_tuner as kt
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print('op determinism unavailable:', e)
print('TF', tf.__version__, '| keras-tuner', kt.__version__)

TF 2.10.0 | keras-tuner 1.4.8


In [2]:
MAT_PATH = 'B0005.mat'
TEST_START_CYCLE = 134          # Tian SP = 80%
HORIZON_TARGET   = 35           # test horizon; validation folds use the same
VAL_ORIGINS      = [89, 94, 99] # each forecasts 35 cycles, ending before cycle 134
SEEDS_SELECT, SEEDS_TRIAL, SEEDS_FINAL = 5, 2, 10
MAX_TRIALS   = 30
PARAM_PENALTY = 2e-4            # added per 100k parameters
EPOCHS, PATIENCE = 300, 30
SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

from scipy.io import loadmat
mat = loadmat(MAT_PATH); cycles = mat['B0005'][0, 0]['cycle'][0]
rows, n = [], 0
for i in range(len(cycles)):
    r = cycles[i]
    if r['type'][0] != 'discharge': continue
    n += 1; d = r['data'][0][0]
    V = d['Voltage_measured'][0].astype(float); T = d['Temperature_measured'][0].astype(float)
    t = d['Time'][0].astype(float)
    tt = lambda vth: (float(t[np.where(V <= vth)[0][0]]) if len(np.where(V <= vth)[0]) else float(t[-1]))
    rows.append(dict(cycle=n, capacity=float(d['Capacity'][0][0]),
                     discharge_time=float(t[-1]), t_to_3p8V=tt(3.8), t_to_3p5V=tt(3.5),
                     mean_voltage=float(V.mean()), min_voltage=float(V.min()),
                     mean_temp=float(T.mean()), max_temp=float(T.max()),
                     temp_rise=float(T.max()-T[0])))
HI = pd.DataFrame(rows); C0 = HI['capacity'].iloc[0]
HI['SOH'] = (HI['capacity']/C0).astype(np.float32)
S, CYC = HI['SOH'].values.astype(np.float32), HI['cycle'].values
IND_COLS = ['capacity','discharge_time','t_to_3p8V','t_to_3p5V','mean_voltage',
            'min_voltage','mean_temp','max_temp','temp_rise']
TEST_START = int(np.where(CYC >= TEST_START_CYCLE)[0][0])
HORIZON    = len(HI) - TEST_START
FLOOR = float(np.sqrt(mean_squared_error(S[TEST_START:], np.full(HORIZON, S[TEST_START-1]))))
print('cycles %d | test: train < %d, forecast %d cycles | trivial floor %.5f'
      % (len(HI), TEST_START_CYCLE, HORIZON, FLOOR))

def features(mode):
    """Returns (matrix, n_predicted_channels). Channel 0 is always SOH."""
    cols = ['capacity'] if mode in ('F0', 'F1') else IND_COLS
    ref  = HI[cols].iloc[0].replace(0, 1.0)
    F    = (HI[cols]/ref).values.astype(np.float32)
    n_pred = F.shape[1]
    if mode in ('F1', 'F3'):
        F = np.column_stack([F, (CYC/100.0).astype(np.float32)])   # known future covariate
    return F.astype(np.float32), n_pred

FOLDS = [(o, HORIZON_TARGET) for o in VAL_ORIGINS]
for o, h in FOLDS:
    oi = int(np.where(CYC >= o)[0][0])
    print('  val fold: train < cycle %d, forecast cycles %d..%d' % (o, o, CYC[min(oi+h, len(HI))-1]))

cycles 168 | test: train < 134, forecast 35 cycles | trivial floor 0.02927
  val fold: train < cycle 89, forecast cycles 89..123
  val fold: train < cycle 94, forecast cycles 94..128
  val fold: train < cycle 99, forecast cycles 99..133


In [3]:
def make_window_data(F, n_pred, cutoff_idx, window):
    X, y = [], []
    for e in range(window, cutoff_idx):
        X.append(F[e-window:e])
        y.append(F[e, :n_pred] - F[e-1, :n_pred])     # delta target, all predicted channels
    X, y = np.asarray(X, np.float32), np.asarray(y, np.float32)
    cut = int(0.85*len(X))
    return X[:cut], y[:cut], X[cut:], y[cut:]

def build(window, n_feat, n_out, filters=32, ms_kernels=(3,5,7), lstm_units=32,
          n_cnn=1, n_dense=1, dense_units=64, dropout=0.1, l2=1e-5,
          lr=1e-3, optimizer='adam', seed=SEED):
    """Previous round's winning architecture: multi-scale CNN + BiLSTM, no attention."""
    reg = regularizers.l2(l2) if l2 > 0 else None
    kw  = dict(kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed),
               kernel_regularizer=reg)
    inp = Input(shape=(window, n_feat)); c = inp
    for _ in range(n_cnn):
        c = Concatenate()([Conv1D(filters, k, padding='same', activation='relu', **kw)(c)
                           for k in ms_kernels])
        c = BatchNormalization()(c)
        c = Dropout(dropout, seed=seed)(c)
    c = GlobalAveragePooling1D()(c)
    r = Bidirectional(LSTM(lstm_units, **kw))(inp)
    h = Concatenate()([c, r])
    for _ in range(n_dense):
        h = Dense(dense_units, activation='relu', **kw)(h)
        h = Dropout(dropout, seed=seed)(h)
    out = Dense(n_out, activation='linear', **kw)(h)
    opt = {'adam': tf.keras.optimizers.Adam,
           'rmsprop': tf.keras.optimizers.RMSprop}[optimizer](lr)
    m = Model(inp, out); m.compile(optimizer=opt, loss='mse'); return m

def fit(m, Xf, yf, Xv, yv, batch=16):
    m.fit(Xf, yf, validation_data=(Xv, yv), epochs=EPOCHS, batch_size=batch, verbose=0,
          shuffle=True,
          callbacks=[EarlyStopping('val_loss', patience=PATIENCE, restore_best_weights=True),
                     ReduceLROnPlateau('val_loss', factor=0.5, patience=10, min_lr=1e-7)])
    return m

def roll(m, F, n_pred, has_cycle, window, origin_idx, horizon):
    """Multivariate autoregressive forecast. Predicted channels are fed back; the
       cycle index, being known for future cycles, is supplied as its true value."""
    win = [F[i].copy() for i in range(origin_idx-window, origin_idx)]
    soh = []
    for k in range(horizon):
        x = np.asarray(win[-window:], np.float32).reshape(1, window, F.shape[1])
        p = m(x, training=False).numpy().ravel()
        nxt = win[-1].copy()
        nxt[:n_pred] = win[-1][:n_pred] + p
        nxt[0] = float(np.clip(nxt[0], 0.0, 1.2))
        if has_cycle:
            idx = origin_idx + k
            nxt[-1] = F[idx, -1] if idx < len(F) else (CYC[-1]+1)/100.0
        win.append(nxt); soh.append(nxt[0])
    return np.asarray(soh, np.float32)

def evaluate(mode, hp_kw, seed, origin_cycle, horizon, batch=16):
    """Train on cycles < origin, forecast `horizon` cycles from it, return RMSE on SOH."""
    F, n_pred = features(mode)
    has_cycle = mode in ('F1', 'F3')
    window    = hp_kw['window']
    oi = int(np.where(CYC >= origin_cycle)[0][0])
    Xf, yf, Xv, yv = make_window_data(F, n_pred, oi, window)
    K.clear_session(); gc.collect()
    np.random.seed(seed); tf.random.set_seed(seed)
    kw = {k: v for k, v in hp_kw.items() if k != 'window'}
    m  = build(window, F.shape[1], n_pred, seed=seed, **kw)
    fit(m, Xf, yf, Xv, yv, batch=batch)
    h  = min(horizon, len(HI)-oi)
    pred = roll(m, F, n_pred, has_cycle, window, oi, h)
    return float(np.sqrt(mean_squared_error(S[oi:oi+h], pred))), m.count_params()

DEFAULTS = dict(window=30, filters=32, ms_kernels=(5,9,13), lstm_units=32,
                n_cnn=1, n_dense=1, dense_units=64, dropout=0.1, l2=1e-5,
                lr=1e-3, optimizer='adam')
print('helpers ready')

helpers ready


## Stage B — which information does the model actually need?

Each feature mode is run on the fold closest to the test setup (origin 99, 35-cycle horizon) for
selection, then on the test itself, 5 seeds each.

In [4]:
t0, fb = time.time(), []
for mode in ('F0', 'F1', 'F2', 'F3'):
    vs, ts, prm = [], [], None
    for s in range(SEEDS_SELECT):
        v, prm = evaluate(mode, DEFAULTS, SEED+s, 99, HORIZON_TARGET)
        t, _   = evaluate(mode, DEFAULTS, SEED+s, TEST_START_CYCLE, HORIZON)
        vs.append(v); ts.append(t)
    fb.append(dict(mode=mode, val_mean=float(np.mean(vs)), val_std=float(np.std(vs, ddof=1)),
                   test_mean=float(np.mean(ts)), test_std=float(np.std(ts, ddof=1)),
                   test_best=float(np.min(ts)), params=int(prm)))
    print('%-3s  val %.5f +/- %.5f | test %.5f +/- %.5f | best %.5f | params %s'
          % (mode, np.mean(vs), np.std(vs, ddof=1), np.mean(ts), np.std(ts, ddof=1),
             np.min(ts), format(prm, ',')))
FB = pd.DataFrame(fb).sort_values('val_mean').reset_index(drop=True)
BEST_MODE = FB.iloc[0]['mode']
print('\nselected on validation: %s' % BEST_MODE)
print('\n(previous round, capacity only, no cycle index: test 0.01053)')
print('total %.0f s' % (time.time()-t0))

F0   val 0.05877 +/- 0.02778 | test 0.06896 +/- 0.02526 | best 0.04510 | params 20,417
F1   val 0.33801 +/- 0.11942 | test 0.22204 +/- 0.16471 | best 0.02436 | params 21,537
F2   val 0.29645 +/- 0.18137 | test 0.30219 +/- 0.14008 | best 0.10003 | params 29,897
F3   val 0.46059 +/- 0.09637 | test 0.43277 +/- 0.12176 | best 0.31369 | params 31,017

selected on validation: F0

(previous round, capacity only, no cycle index: test 0.01053)
total 1220 s


## Stage A — constrained Bayesian search

Twelve hyperparameters. Objective is the mean forecast RMSE across the three validation origins,
each with the same 35-cycle horizon as the test, plus a penalty on model size.

In [5]:
OBJ = 'val_folds_penalised'

def hp_kwargs(hp):
    return dict(window      = hp.Choice('window', [10, 20, 30, 40]),
                n_cnn       = hp.Int('n_cnn', 1, 2),
                filters     = hp.Int('filters', 8, 64, step=8),
                ms_kernels  = tuple(int(v) for v in hp.Choice(
                                  'ms_kernels', ['3,5,7', '5,9,13', '3,7,11']).split(',')),
                lstm_units  = hp.Int('lstm_units', 8, 64, step=8),
                n_dense     = hp.Int('n_dense', 1, 2),
                dense_units = hp.Int('dense_units', 16, 128, step=16),
                dropout     = hp.Float('dropout', 0.0, 0.4, step=0.05),
                l2          = hp.Float('l2', 1e-7, 1e-2, sampling='log'),
                lr          = hp.Float('lr', 1e-4, 5e-3, sampling='log'),
                optimizer   = hp.Choice('optimizer', ['adam', 'rmsprop']))

class ConstrainedBO(kt.BayesianOptimization):
    def run_trial(self, trial, *a, **kw):
        hp = trial.hyperparameters
        kwargs = hp_kwargs(hp); batch = hp.Choice('batch_size', [8, 16, 32])
        scores, prm = [], 0
        for origin, hz in FOLDS:
            for s in range(SEEDS_TRIAL):
                try:
                    v, prm = evaluate(BEST_MODE, kwargs, SEED+s, origin, hz, batch=batch)
                    if not np.isfinite(v): v = 1e3
                except Exception as e:
                    print('   fail (%s)' % type(e).__name__); v = 1e3
                scores.append(v)
        base = float(np.mean(scores))
        pen  = PARAM_PENALTY * (prm/1e5)
        print('   trial %s: folds %.5f + penalty %.5f = %.5f  (%s params)'
              % (trial.trial_id, base, pen, base+pen, format(prm, ',')))
        return {OBJ: base + pen}

t0 = time.time()
tuner = ConstrainedBO(hypermodel=lambda hp: build(hp_kwargs(hp)['window'], 1, 1),
                      objective=kt.Objective(OBJ, direction='min'),
                      max_trials=MAX_TRIALS, seed=SEED,
                      directory='bo_round2_b0005', project_name=BEST_MODE, overwrite=False)
tuner.search()
best_hp = tuner.get_best_hyperparameters(1)[0]
print('\nsearch done in %.0f s' % (time.time()-t0))
for k, v in sorted(best_hp.values.items()): print('   %-14s %s' % (k, v))

Trial 30 Complete [00h 01m 28s]
val_folds_penalised: 0.20435251301155122

Best val_folds_penalised So Far: 0.010133247018608729
Total elapsed time: 02h 13m 09s

search done in 7990 s
   batch_size     8
   dense_units    112
   dropout        0.15000000000000002
   filters        32
   l2             0.0032145190818948744
   lr             0.0034730507857514727
   lstm_units     16
   ms_kernels     3,7,11
   n_cnn          2
   n_dense        1
   optimizer      adam
   window         10


In [6]:
kwargs = hp_kwargs(best_hp); batch = best_hp.values.get('batch_size', 16)
scores, prm = [], None
for s in range(SEEDS_FINAL):
    v, prm = evaluate(BEST_MODE, kwargs, SEED+1000+s, TEST_START_CYCLE, HORIZON, batch=batch)
    scores.append(v)
TUNED = dict(mode=BEST_MODE, mean=float(np.mean(scores)), std=float(np.std(scores, ddof=1)),
             best=float(np.min(scores)), worst=float(np.max(scores)), params=int(prm),
             per_seed=[round(v, 5) for v in scores], hp=dict(best_hp.values))
PREV_UNTUNED, PREV_TUNED = 0.01053, 0.01063
print('='*72)
print('round 1  untuned (capacity only)      : %.5f' % PREV_UNTUNED)
print('round 1  tuned, 46 hyperparams        : %.5f' % PREV_TUNED)
print('round 2  features %s, untuned         : %.5f' % (BEST_MODE, FB.iloc[0]['test_mean']))
print('round 2  features %s + constrained BO : %.5f +/- %.5f  (best seed %.5f)'
      % (BEST_MODE, TUNED['mean'], TUNED['std'], TUNED['best']))
print('params %s | trivial floor %.5f | skill %+.3f'
      % (format(prm, ','), FLOOR, 1-TUNED['mean']/FLOOR))
print('='*72)

round 1  untuned (capacity only)      : 0.01053
round 1  tuned, 46 hyperparams        : 0.01063
round 2  features F0, untuned         : 0.06896
round 2  features F0 + constrained BO : 0.01567 +/- 0.00007  (best seed 0.01557)
params 83,009 | trivial floor 0.02927 | skill +0.465


## Stage C — does the variance finding still hold?

In [7]:
PUB = [('CNN-BiLSTM-AM','Tian',0.00481),('CNN-BiLSTM','Tian',0.00595),('BiLSTM','Tian',0.00620),
       ('1D CNN-BiLSTM','Gao',0.00772),('CNN','Tian',0.00863),('1D CNN','Gao',0.00866),
       ('1D CNN-LSTM','Gao',0.00932),('1D CNN-RNN','Gao',0.01661),('RNN','Gao',0.01776),
       ('BiLSTM','Gao',0.01986),('LSTM','Gao',0.02302)]
pub_lo, pub_hi = min(v for _,_,v in PUB), max(v for _,_,v in PUB)
spread = TUNED['worst'] - TUNED['best']
print('published range across 11 models : %.5f .. %.5f  (spread %.5f)' % (pub_lo, pub_hi, pub_hi-pub_lo))
print('our tuned model, 10 seeds        : %.5f .. %.5f  (spread %.5f)'
      % (TUNED['best'], TUNED['worst'], spread))
print('per-seed: %s' % TUNED['per_seed'])
print()
print('VERDICT: seed spread of ONE architecture %s the entire published range'
      % ('still EXCEEDS' if spread > (pub_hi-pub_lo) else 'is now SMALLER than'))

rows = [dict(method=m, source=s, rmse=v) for m, s, v in PUB]
rows += [dict(method='Ours round 1 (capacity only, tuned)', source='this work', rmse=PREV_TUNED),
         dict(method='Ours round 2 (%s + constrained BO)' % BEST_MODE, source='this work',
              rmse=round(TUNED['mean'], 5)),
         dict(method='   our best seed', source='this work', rmse=round(TUNED['best'], 5)),
         dict(method='   trivial floor', source='reference', rmse=round(FLOOR, 5))]
B = pd.DataFrame(rows).sort_values('rmse').reset_index(drop=True); B.index = B.index+1
pd.set_option('display.width', 200)
print('\n' + B.to_string())
B.to_csv('benchmark_b0005_round2.csv')
FB.to_csv('feature_ablation_b0005.csv', index=False)
json.dump(dict(floor=FLOOR, folds=FOLDS, feature_ablation=fb, tuned=TUNED,
               round1=dict(untuned=PREV_UNTUNED, tuned=PREV_TUNED)),
          open('round2_results_b0005.json','w'), indent=2, default=str)
print('\nsaved benchmark_b0005_round2.csv, feature_ablation_b0005.csv, round2_results_b0005.json')

published range across 11 models : 0.00481 .. 0.02302  (spread 0.01821)
our tuned model, 10 seeds        : 0.01557 .. 0.01583  (spread 0.00026)
per-seed: [0.01557, 0.01569, 0.01565, 0.01569, 0.01583, 0.0157, 0.01561, 0.01567, 0.01571, 0.01562]

VERDICT: seed spread of ONE architecture is now SMALLER than the entire published range

                                 method     source     rmse
1                         CNN-BiLSTM-AM       Tian  0.00481
2                            CNN-BiLSTM       Tian  0.00595
3                                BiLSTM       Tian  0.00620
4                         1D CNN-BiLSTM        Gao  0.00772
5                                   CNN       Tian  0.00863
6                                1D CNN        Gao  0.00866
7                           1D CNN-LSTM        Gao  0.00932
8   Ours round 1 (capacity only, tuned)  this work  0.01063
9                         our best seed  this work  0.01557
10   Ours round 2 (F0 + constrained BO)  this work  0.01567
11    